# Stage 2a — Continued Pre-Training (FINAL CLEAN)

Trendyol-LLM-7b-base-v1.0 → 387M token Türkçe akademik continued pre-train.

**Önce:** Runtime → Disconnect and delete runtime (eski state temizlensin) → yeni runtime A100.

**Çalıştırma:**
1. Cell 1 (env + minimal install) — bir kez
2. **Runtime → Restart session** — zorunlu
3. Cell 2 (env tekrar) → Cell 11'e kadar sırayla

**Recipe:** LR=1e-5, effective batch=32, bf16 + paged_adamw_8bit + gradient_checkpointing, max_seq=2048, 1 epoch, Drive resume.

**Pin'ler:** transformers 4.45.2 (Trendyol repo 4.46+ 404), bitsandbytes 0.43.3 (Colab CUDA 12). Diğer paketlere DOKUNMUYORUZ (numpy/pandas/scipy/pyarrow Colab default = ABI uyumlu).

## 1. Env + Install (BİR KEZ → RESTART)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi
# NO --force-reinstall: Colab's numpy/pandas/scipy/pyarrow are ABI-consistent.
# bitsandbytes intentionally OMITTED — it's broken on current Colab (triton.ops
# removed, CUDA 13 binaries missing). We use pure-PyTorch Adafactor in Cell 10
# instead, which gives similar memory savings.
!pip install -q \
  "transformers==4.45.2" "tokenizers>=0.20,<0.21" \
  "datasets>=2.20,<3.0" "accelerate>=0.34,<1.0" \
  sentencepiece

print('\n*** RESTART SESSION NOW (Runtime → Restart session) ***')
print('After restart: skip Cell 1, run Cell 2 onwards.')

## 2. Env (restart sonrası tekrar set)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('CUDA alloc conf:', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))

## 3. Workspace

In [ ]:
WORK_DIR = '/content/trakad_pretrain'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)
!df -h /content | tail -1

## 4. HF auth

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 5. Mount Drive (persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/trakad-7b-base'
os.makedirs(DRIVE_OUT, exist_ok=True)
ckpts = sorted([d for d in os.listdir(DRIVE_OUT) if d.startswith('checkpoint-')])
print(f'Drive checkpoints found: {ckpts if ckpts else "none — will start fresh"}')

## 6. Pull corpus from HF Hub

In [ ]:
from huggingface_hub import hf_hub_download
import json

TRAIN_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/train.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
VAL_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/val.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
STATS = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/_corpus_stats.json',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
stats = json.load(open(STATS))
for k, v in stats.items():
    print(f'  {k:25s} {v}')
print(f'\nTrain parquet: {os.path.getsize(TRAIN_PARQUET)/1024/1024:.0f} MB')
print(f'Val parquet:   {os.path.getsize(VAL_PARQUET)/1024/1024:.0f} MB')

## 7. Build HF Datasets

In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_parquet(TRAIN_PARQUET)
val_df = pd.read_parquet(VAL_PARQUET)
print(f'Train chunks: {len(train_df):,}')
print(f'Val chunks:   {len(val_df):,}')

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

def add_labels(ex):
    ex['labels'] = ex['input_ids']
    return ex

train_ds = train_ds.map(add_labels, num_proc=4)
val_ds = val_ds.map(add_labels, num_proc=4)
print(f'\nFirst chunk length: {len(train_ds[0]["input_ids"])}')

## 8. Load Trendyol-7B base (bf16 + grad ckpt)

In [ ]:
import torch
from transformers import LlamaTokenizer, AutoModelForCausalLM

BASE = 'Trendyol/Trendyol-LLM-7b-base-v1.0'
tokenizer = LlamaTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'tokenizer: vocab={tokenizer.vocab_size}, eos={tokenizer.eos_token_id}')

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
print(f'model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params on {next(model.parameters()).device}')

## 9. torch.load weights_only patch (PyTorch 2.6+)

In [ ]:
_orig_load = torch.load
def _patched_load(*a, **k):
    k.setdefault('weights_only', False)
    return _orig_load(*a, **k)
torch.load = _patched_load
print('torch.load patched: weights_only=False default')

## 10. Full training (Drive resume otomatik, ~38h A100)

In [ ]:
import math, time
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers.optimization import Adafactor

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

OUT_DIR = DRIVE_OUT
EFFECTIVE_BATCH = 32
EPOCHS = 1

steps_per_epoch = math.ceil(len(train_ds) / EFFECTIVE_BATCH)
print(f'Output dir: {OUT_DIR}')
print(f'Steps/epoch: {steps_per_epoch:,}')

# Drop optimizer.pt from old checkpoints (saved as paged_adamw_8bit, incompatible
# with Adafactor's param group layout). Model weights stay.
for ckpt in sorted(os.listdir(OUT_DIR)) if os.path.isdir(OUT_DIR) else []:
    if ckpt.startswith('checkpoint-'):
        p = os.path.join(OUT_DIR, ckpt, 'optimizer.pt')
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed stale {ckpt}/optimizer.pt')

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=EFFECTIVE_BATCH,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=20,
    save_strategy='steps',
    save_steps=1000,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=1000,
    report_to=[],
    gradient_checkpointing=True,
    lr_scheduler_type='cosine',
    dataloader_num_workers=2,
)

# Manual Adafactor — HF Trainer's 'optim=adafactor' is broken (missing beta1 in
# param_group). bitsandbytes 8-bit is broken on Colab too (CUDA 13 / triton.ops
# missing). Adafactor pure-PyTorch is the working path, ~14 GB optimizer state
# vs AdamW's 56 GB — comfortable on A100 80GB.
optimizer = Adafactor(
    model.parameters(),
    lr=1e-5,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    optimizers=(optimizer, None),
)

resume = os.path.isdir(OUT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(OUT_DIR)
)
print(f'Resume from checkpoint: {resume}')

t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
dt = time.time() - t0
print(f'\nFull training done in {dt/3600:.2f} hours -> {OUT_DIR}')

tokenizer.save_pretrained(OUT_DIR)
print('Tokenizer saved alongside model.')

## 11. Final eval + generations + push

In [ ]:
eval_metrics = trainer.evaluate()
print(f'Final val loss: {eval_metrics["eval_loss"]:.4f}')
print(f'Final val PPL:  {math.exp(eval_metrics["eval_loss"]):.2f}')

prompts = [
    'Bu çalışmada, derin öğrenme yöntemleri kullanılarak Türkçe metinler üzerinde',
    'Yenilenebilir enerji kaynaklarından rüzgar enerjisinin Türkiye\'deki potansiyeli',
    'Akademik araştırmalarda doğal dil işleme teknikleri',
]
model.eval()
for p in prompts:
    inputs = tokenizer(p, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=True, top_p=0.9, temperature=0.7)
    print(f'\n=== {p}')
    print(tokenizer.decode(out[0], skip_special_tokens=True))

from huggingface_hub import HfApi
REPO = 'hakansabunis/trakad-7b-base'
api = HfApi(token=hf_token)
api.create_repo(REPO, repo_type='model', exist_ok=True, private=False, token=hf_token)
print(f'\nUploading {OUT_DIR} -> {REPO}')
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=REPO,
    repo_type='model',
    commit_message='Trakad-7B-base — continued pre-train of Trendyol-LLM-7b-base-v1.0 on 387M tokens Turkish academic abstracts',
    token=hf_token,
    allow_patterns=['*.safetensors', '*.json', '*.bin', 'tokenizer*', '*.txt', '*.model'],
)
print(f'Done: https://huggingface.co/{REPO}')